# Finding stars and apature photometry with anulus backgrounds

* 이 노트북을 구글 코랩에서 실행하고자 한다면 [파일] - [드라이브에 사본 저장]을 하여 본인의 소유로 만든 후에 코드를 실행하거나 수정할 수 있습니다.

* 이 파일은 실제 수업에 사용하므로 필요에 따라 예고 없이 변경될 수 있습니다.

* If you have any questions or comments on this document, please email me(Kiehyun.Park@gmail.com).

* 이 파일(문서)는 공교육 현장에서 수업시간에 자유롭게 사용할 수 있으나, 다른 목적으로 사용할 시에는 사전에 연락을 주셔서 상의해 주시기 바랍니다.



천체 관측 중 CCD(charge couple device) 관측 자료를 이용하여 별을 찾고, 구경 측광 하는 방법을 다룹니다.

## 필요한 환경

이 프로젝트를 위해서는 아래의 모듈이 필요합니다.

> numpy, matplotlib, ccdproc, astropy, photutils, version_information



### 구글 코랩에 한글 폰트 설치

matplotlib에서 한글을 사용하기 위해서는 한글 폰트가 필요합니다. 구글 코랩에서 현재의 Jupyter notebook을 실행한다면 아래 코드를 실행 한 후 런타임 다시 시작을 해 줘야 한글을 사용할 수 있을 것입니다.

In [1]:
# !sudo apt-get install -y fonts-nanum
# !sudo fc-cache -fv
# !rm ~/.cache/matplotlib -rf

### 런타임 다시 시작

위의 셀을 실행한 다음 반드시 다음 과정을 잊지 마시기 바랍니다.

* [메뉴]-[런타임]-[런터임 다시 시작]

* [메뉴]-[런타임]-[이전 셀 실행]을 해주어야 합니다.

### 한글 폰트 사용

위에서 한글 폰트를 설치하고, 런타임 다시시작을 했다면 구글 코랩에서 폰트 경로를 설정하여 한글 사용이 가능해 집니다.

In [2]:
#visualization
import matplotlib as mpl
import matplotlib.pylab as plb
import matplotlib.pyplot as plt

# 브라우저에서 바로 그려지도록
%matplotlib inline

# 그래프에 retina display 적용
%config InlineBackend.figure_format = 'retina'

# Colab 의 한글 폰트 설정
#plt.rc('font', family='NanumBarunGothic')

# 유니코드에서  음수 부호설정
mpl.rc('axes', unicode_minus=False)

### 모듈 설치 및 버전 확인

아래 셀을 실행하면 이 노트북을 실행하는데 필요한 모듈을 설치하고 파이썬 및 관련 모듈의 버전을 확인할 수 있습니다.

In [3]:
import importlib, sys, subprocess
packages = "numpy, matplotlib, ccdproc, astropy, gdown, photutils, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"**** {pkg} module is now being installed.")
    else:
        print(f"******** {pkg} module is already installed.")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

******** numpy module is already installed.
******** matplotlib module is already installed.
******** ccdproc module is already installed.
******** astropy module is already installed.
******** gdown module is already installed.
******** photutils module is already installed.
******** version_information module is already installed.
This notebook was generated at 2023-11-19 09:41:44 (KST = GMT+0900) 
0 Python     3.11.5 64bit [GCC 11.2.0]
1 IPython    8.15.0
2 OS         Linux 5.15.0 88 generic x86_64 with glibc2.31
3 numpy      1.26.0
4 matplotlib 3.8.0
5 ccdproc    2.4.0
6 astropy    5.3.4
7 gdown      4.7.1
8 photutils  1.6.0
9 version_information 1.0.4


## 데이터 저장

### FITS 파일 저장 폴더 생성

FITS 파일을 저장할 폴더를 "Aperture_Photometry" 이라는 이름으로 생성해 보겠습니다.

* 만약 리눅스 시스템 이라면 shell 명령어로 가능한데, "!"를 붙이면 shell 명령어를 실행할 수 있습니다.
> !mkdir Aperture_Photometry

OS의 영향을 받지 않기 위하여 pathlib을 사용하여 폴더를 생성하겠습니다.

In [4]:
import os
from pathlib import Path
BASEPATH = Path("./")
save_dir_name = "Aperture_Photometry"
print(f"BASEPATH: {BASEPATH}")

if not (BASEPATH/save_dir_name).exists():
    os.mkdir(str(BASEPATH/save_dir_name))
    print (f"{str(BASEPATH/save_dir_name)} is created...")
else :
    print (f"{str(BASEPATH/save_dir_name)} is already exist...")

BASEPATH: .
Aperture_Photometry is created...


### FITS 파일 다운로드

나의 구글 드라이브에 저장된 CCD 관측 파일을 다운받아 보겠습니다.

GNU Wget은 HTTP 통신 또는 FTP 통신을 사용해 서버에서 파일 또는 콘텐츠를 다운로드할 때 사용하는 소프트웨어입니다. Wget의 특징은 여러 파일을 한 번에 다운로드하거나 웹 페이지의 링크를 순회하며 여러 콘텐츠를 자동으로 다운로드할 때 매우 편리합니다.

MS Windows에서는 별도로 설치를 해줘야 하며,
구글 코랩도 wget을을 지원해주니 아래 셀을 실행하면 자신의 [작업 영역]에 저장된다. 하지만 이 파일은 이 런타임이 재실행 될때는 삭제됨에 유의하시기 바랍니다.

아래 코드를 실행하면 여러분의 작업영역에 해당 파일을 저장해 줄 것입니다.



In [5]:
fname1 = "M35-g3035786.fits"
fid1 = "1GgWOwmSk0LJkJlDQc5SnJtHVWqbLSsrj"
fname2 = "reduced_M35-g3035786.fits"
fid2 = "15oWP8MrQ3-_S3lTQpUXzoS5jtj3pfojO"

# wget을 이용(나의 구글드라이브에서 공유한 파일을 구글 코랩에서 사용할 경우)
# !wget --no-check-certificate "https://docs.google.com/uc?export=download&id={fid1}" -O {save_dir_name}/{fname1}
# !wget --no-check-certificate "https://docs.google.com/uc?export=download&id={fid2}" -O {save_dir_name}/{fname2}

# gdown을 이용(나의 구글드라이브에서 공유한 파일을 다운로드)
!gdown {fid1} -O {save_dir_name}/{fname1}
!gdown {fid2} -O {save_dir_name}/{fname2}

/bin/bash: gdown: command not found
/bin/bash: gdown: command not found


### 데이터 확인

* 코랩을 사용할 경우에는 오른쪽의 [파일]창에서 확인할 수 있습니다.
* linux shell 명령어로 다음과 같이 확인해 볼 수 있습니다.
> !ls -l Aperture_Photometry

OS의 영향을 받지 않고 파이썬으로 확인하는 방법은 아래와 같이 하면 됩니다.

In [6]:
!ls -l Aperture_Photometry

total 0


In [7]:
fpaths = sorted(list((BASEPATH/save_dir_name).glob('*.fit*')))
print(f"fpaths: {fpaths}")
print(f"len(fpaths): {len(fpaths)}")

fpaths: []
len(fpaths): 0


##  FITS handling

### Load files

fits file을  읽어 확인해 보겠습니다.

In [8]:
from astropy.io import fits

fpath = Path(fpaths[0])
print(f"fpath: {fpath}")
print(f"type(fpath): {type(fpath)}")

hdul = fits.open(str(fpath), unit="adu")

print("type(hdul) :", type(hdul))
print("type(hdul[0]) :", type(hdul[0]))

IndexError: list index out of range

### header

ccds라는 이름에 HDUList들이 리스트 형태로 들어 있습니다. 각각의 hdulist는 2차원 이므로 index는 [0]번만 존재합니다.

In [ ]:
print("type(hdul[0].hedaer) :", type(hdul[0].header))
hdul[0].header

header는 key와 value가 들어 있습니다.

In [ ]:
print("hdul[0].hedaer['DATE-OBS'] :", hdul[0].header['DATE-OBS'])
print("type(hdul[0].hedaer['DATE-OBS']) :", type(hdul[0].header['DATE-OBS']))

### data

관측 자료는 numpy.ndarray 형태로 들어 있음을 알 수 있습니다.

In [ ]:
print("type(hdul[0].data) :", type(hdul[0].data))
print("hdul[0].data.dtype :", hdul[0].data.dtype)
print("hdul[0].data.shape :", hdul[0].data.shape)
print("hdul[0].data :", hdul[0].data)

## 별 찾기 (Finding stars)

### 환경 변수 설정

In [ ]:
#######################################################
# Initial guess of FWHM in pixel
FWHM_INIT = 6

# Photometry parameters
R_AP = 1.5 * FWHM_INIT # Aperture radius
R_IN = 4 * FWHM_INIT   # Inner radius of annulus
R_OUT = 6 * FWHM_INIT  # Outer radius of annulus
#######################################################

### DAOstar finder

[(photutils.detection)](https://photutils.readthedocs.io/en/stable/detection.html)을 이용하여 별을 찾아낼 수 있습니다. 그러기 위해서는 별의 FWHM과 배경하늘 한계값(threshold)을 설정해야 한다.


#### threshold1

첫번째로 photutils.segmentation.detect_threshold 함수를 이용하는 것입니다.

In [ ]:
from photutils.segmentation import detect_threshold
thresh = detect_threshold(data=hdul[0].data, nsigma=3)
print(type(thresh))
thresh = thresh[0][0]
print(f'detect_threshold :{thresh:.03f}')

#### threshold2

astropy.stats.sigma_clipped_stats

In [ ]:
from astropy.stats import sigma_clipped_stats

avg, med, std = sigma_clipped_stats(hdul[0].data)  # by default, 3-sigma 5-iteration.
thresh = 5. * std

print(f"avg : {avg:.03f}, med: {med:.03f}, std: {std:.03f}, thresh: {thresh:.03f}")

### DAO result

매개변수에 따라 찾아지는 별의 갯수가 달라지게 됩니다.

In [ ]:
from photutils.detection import DAOStarFinder

fpath = fpaths[0]
print("fpath", (fpath))

hdul =  fits.open(str(fpath), unit="adu")

FWHM = FWHM_INIT

DAOfind = DAOStarFinder(
                        fwhm = FWHM,
                        threshold = thresh,
                        # sharplo = 0.2, sharphi = 1.0,  # default values: sharplo=0.2, sharphi=1.0,
                        # roundlo = 0, roundhi = 1.0,  # default values -1 and +1
                        # sigma_radius = 3,           # default values 1.5
                        # ratio = 1.0,                  # 1.0: circular gaussian
                        # exclude_border = True         # To exclude sources near edges
                        )

DAOfound = DAOfind(hdul[0].data)
print("len(DAOfound) :",len(DAOfound))

#### astropy.table

DAOfound의 결과는 astropy.table로 반환됩니다.

In [ ]:
print("type(DAOfound) :", type(DAOfound))
print(DAOfound.colnames)
DAOfound

#### csv 저장하기

In [ ]:
 DAOfound.write(f"{BASEPATH / save_dir_name}/{fpath.stem}_DAOStarfinder_fwhm_{FWHM}.csv",
                            overwrite = True,
                            format='ascii.fast_csv')

#### daraframe으로 저장

In [ ]:
df = DAOfound.to_pandas()
print(type(df))
df

## tag stars on image

### 이미지에서 별의 픽셀 좌표

In [ ]:
import numpy as np
pos = np.transpose((DAOfound['xcentroid'], DAOfound['ycentroid']))
pos

### 이미지에서 별의 aperture

In [ ]:
from photutils.aperture import CircularAperture as CAp
apert = CAp(pos, r=R_AP)
apert

### 이미지에서 별의 annulus

In [ ]:
from photutils.aperture import CircularAnnulus as CAn

annul = CAn(positions=pos, r_in=4*FWHM, r_out=6*FWHM)
annul

### tag stars on image

In [ ]:
from photutils.aperture import CircularAperture as CAp
from photutils.aperture import CircularAnnulus as CAn

apert = CAp(pos, r=R_AP)
annul = CAn(positions=pos, r_in=4*FWHM, r_out=6*FWHM)

fig, axs = plt.subplots(1, 1, figsize=(12, 12),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs.imshow(hdul[0].data,
            vmax = hdul[0].data.mean()*1.2,
            origin='lower',
            )

apert.plot(color='r', lw=1, alpha=0.5)
annul.plot(color='y', lw=1, alpha=0.7)

###########################################################
# input some text for explaination.
plt.title("Result of DAOStarfinder", fontsize = 14,
    ha='center')

plt.annotate(f'filename: {fpath.stem}', fontsize=8,
    xy=(1, 0), xytext=(-10, -30), va='top', ha='right',
    xycoords='axes fraction', textcoords='offset points')

plt.annotate(f'FWHM: {FWHM}', fontsize=8,
    xy=(0, 0), xytext=(-10, -30), va='top', ha='left',
    xycoords='axes fraction', textcoords='offset points')

plt.annotate(f'Sky threshold: {thresh:.02f}', fontsize=8,
    xy=(0, 0), xytext=(-10, -40), va='top', ha='left',
    xycoords='axes fraction', textcoords='offset points')

plt.annotate(f'Number of star(s): {len(DAOfound)}', fontsize=8,
    xy=(0, 0), xytext=(-10, -50), va='top', ha='left',
    xycoords='axes fraction', textcoords='offset points')

plt.colorbar(im1,
            ax=axs,
            fraction=0.0455, pad=0.04)
plt.tight_layout()

## aperture photometry



### 별 하나씩 확인

In [ ]:
# since our `annul` has many elements,
mask_apert = (apert.to_mask(method='center'))
mask_annul = (annul.to_mask(method='center'))
# CAUTION!! YOU MUST USE 'center', NOT 'exact'!!!

fig = plt.figure(figsize=(12, 12))
for i in range(1, 5):
    for j in range(1, 5):
        cutimg = mask_annul[4*(i-1)+j].cutout(hdul[0].data)

        ax1 = fig.add_subplot(4, 4, 4*(i-1)+j)

        im1 = ax1.imshow(cutimg,
                    vmax = hdul[0].data.mean()*1.2,
                    #origin='lower',
                    )
        ap = CAp(positions = (int(cutimg.shape[0]/2),int(cutimg.shape[1]/2)),
                r=R_AP)
        an = CAn(positions = (int(cutimg.shape[0]/2),int(cutimg.shape[1]/2)),
                r_in=R_IN,
                r_out=R_OUT)
        ax1.grid(ls=':')
        ap.plot(color='r', lw=1, alpha=0.5)
        an.plot(color='y', lw=1, alpha=0.7)

        plt.colorbar(im1,
                    fraction=0.0455, pad=0.04)


plt.tight_layout()
plt.show();

### 20번째 별 선택

In [ ]:
# since our `annul` has many elements,
mask_apert = (apert.to_mask(method='center'))
mask_annul = (annul.to_mask(method='center'))
# CAUTION!! YOU MUST USE 'center', NOT 'exact'!!!

#let me use [20] to use only the annulus
idx=20
cutimg = mask_annul[idx].cutout(hdul[0].data)

fig = plt.figure(figsize=(5, 5))
ax1 = fig.add_subplot(1,1,1)

im1 = ax1.imshow(cutimg,
            vmax = hdul[0].data.mean()*1.2,
            origin='lower',
            )
ap = CAp(positions = (int(cutimg.shape[0]/2),int(cutimg.shape[1]/2)),
        r=R_AP)
an = CAn(positions = (int(cutimg.shape[0]/2),int(cutimg.shape[1]/2)),
        r_in=R_IN,
        r_out=R_OUT)
ax1.grid(ls=':')
ap.plot(color='r', lw=1, alpha=0.5)
an.plot(color='y', lw=1, alpha=0.7)

plt.colorbar(im1,
            fraction=0.0455, pad=0.04)

plt.show()

### aperture area

In [ ]:
from photutils.aperture import aperture_photometry as apphot

fig = plt.figure(figsize=(10, 5))

ax1 = fig.add_subplot(1,2,1)
im1 = ax1.imshow(mask_apert[idx],
            origin='lower',
            )

ax1.grid(ls=':')
ax1.set_title("aperture mask array")

plt.colorbar(im1,
            fraction=0.0455, pad=0.04)

apert_weighted = mask_apert[idx].multiply(hdul[0].data)

ax2 = fig.add_subplot(1,2,2)
im2 = ax2.imshow(apert_weighted,
            vmax = hdul[0].data.mean()*1.2,
            origin='lower',
            )
ax2.grid(ls=':')
ax2.set_title("aperture area")
plt.colorbar(im2,
            fraction=0.0455, pad=0.04)

# 구경 안의 픽셀값
apphot_result = apphot(hdul[0].data, apert, method='center')

# 구경의 넓이
ap_area   = apert.area

plt.annotate(f"aperture_sum: {apphot_result[idx]['aperture_sum']}", fontsize=8,
    xy=(0, 0), xytext=(-10, -25), va='top', ha='left',
    xycoords='axes fraction', textcoords='offset points')

plt.annotate(f"aperture_area: {ap_area:.02f}", fontsize=8,
    xy=(0, 0), xytext=(-10, -40), va='top', ha='left',
    xycoords='axes fraction', textcoords='offset points')

plt.show()

### annulus area

In [ ]:
fig = plt.figure(figsize=(10, 5))

ax1 = fig.add_subplot(1,2,1)
im1 = ax1.imshow(mask_annul[idx],
            origin='lower',
            )

ax1.grid(ls=':')
ax1.set_title(f"annulus mask array(star #{idx})")

plt.colorbar(im1,
            fraction=0.0455, pad=0.04)

annul_weighted = mask_annul[idx].multiply(hdul[0].data)

ax2 = fig.add_subplot(1,2,2)
im2 = ax2.imshow(annul_weighted,
            vmax = hdul[0].data.mean()*1.2,
            origin='lower',
            )
ax2.grid(ls=':')
ax2.set_title(f"annulus area(star #{idx})")
plt.colorbar(im2,
            fraction=0.0455, pad=0.04)

plt.show()

### sky estimation

sky value를 구하는 알고리즘은 여러가지가 있다. 이를 함수로 만들어 사용하자.

In [ ]:
import numpy as np
from astropy.stats import sigma_clip

def sky_fit(all_sky, method='mode', sky_nsigma=3, sky_iter=5, \
            mode_option='sex', med_factor=2.5, mean_factor=1.5):
    '''
    Estimate sky from given sky values.
    Parameters
    ----------
    all_sky : ~numpy.ndarray
        The sky values as numpy ndarray format. It MUST be 1-d for proper use.
    method : {"mean", "median", "mode"}, optional
        The method to estimate sky value. You can give options to "mode"
        case; see mode_option.
        "mode" is analogous to Mode Estimator Background of photutils.
    sky_nsigma : float, optinal
        The input parameter for sky sigma clipping.
    sky_iter : float, optinal
        The input parameter for sky sigma clipping.
    mode_option : {"sex", "IRAF", "MMM"}, optional.
        sex  == (med_factor, mean_factor) = (2.5, 1.5)
        IRAF == (med_factor, mean_factor) = (3, 2)
        MMM  == (med_factor, mean_factor) = (3, 2)
    Returns
    -------
    sky : float
        The estimated sky value within the all_sky data, after sigma clipping.
    std : float
        The sample standard deviation of sky value within the all_sky data,
        after sigma clipping.
    nsky : int
        The number of pixels which were used for sky estimation after the
        sigma clipping.
    nrej : int
        The number of pixels which are rejected after sigma clipping.
    -------
    '''
    sky = all_sky.copy()
    if method == 'mean':
        return np.mean(sky), np.std(sky, ddof=1)

    elif method == 'median':
        return np.median(sky), np.std(sky, ddof=1)

    elif method == 'mode':
        sky_clip   = sigma_clip(sky, sigma=sky_nsigma,
                                maxiters=sky_iter, #iters=sky_iter,
                                )
        sky_clipped= sky[np.invert(sky_clip.mask)]
        nsky       = np.count_nonzero(sky_clipped)
        mean       = np.mean(sky_clipped)
        med        = np.median(sky_clipped)
        std        = np.std(sky_clipped, ddof=1)
        nrej       = len(all_sky) - len(sky_clipped)

        if nrej < 0:
            raise ValueError('nrej < 0: check the code')

        if nrej > nsky: # rejected > survived
            raise Warning('More than half of the pixels rejected.')

        if mode_option == 'IRAF':
            if (mean < med):
                sky = mean
            else:
                sky = 3 * med - 2 * mean

        elif mode_option == 'MMM':
            sky = 3 * med - 2 * mean

        elif mode_option == 'sex':
            if (mean - med) / std > 0.3:
                sky = med
            else:
                sky = (2.5 * med) - (1.5 * mean)
        else:
            raise ValueError('mode_option not understood')

        return sky, std, nsky, nrej

annulus 부분만 sky_pixel로 저장하여 sky value를 구해보자.

In [ ]:
sky_non0   = np.nonzero(annul_weighted)
sky_pixel  = annul_weighted[sky_non0]

msky, stdev, nsky, nrej = sky_fit(sky_pixel, method='mode', mode_option='sex')
print(f"msky: {msky:.03f}, stdev: {stdev:.03f}, nsky: {nsky}, nrej: {nrej}")

In [ ]:
fig = plt.figure(figsize=(8, 8))

ax1 = fig.add_subplot(2,2,1)
im1 = ax1.imshow(mask_annul[idx],
            origin='lower',
            )

ax1.grid(ls=':')
ax1.set_title(f"annulus mask array(star #{idx})")

plt.colorbar(im1,
            fraction=0.0455, pad=0.04)

ax2 = fig.add_subplot(2,2,2)
im2 = ax2.imshow(annul_weighted,
            vmax = hdul[0].data.mean()*1.2,
            origin='lower',
            )
ax2.grid(ls=':')
ax2.set_title(f"annulus area (star #{idx})")
plt.colorbar(im2,
            fraction=0.0455, pad=0.04)

ax3 = plt.subplot(2, 2, 3)
ax3.set_title(f'The histrogram of sky value (star #{idx})')
ax3.hist(sky_pixel, 50, histtype='step')
ax3.axvline(msky, ls=':', color='r')

plt.annotate(f"mean of sky: {msky:.02f}", fontsize=8,
    xy=(0, 0), xytext=(-10, -25), va='top', ha='left',
    xycoords='axes fraction', textcoords='offset points')

plt.annotate(f"pix number of sky: {nsky}, rejection pix number {nrej}, ", fontsize=8,
    xy=(0, 0), xytext=(-10, -40), va='top', ha='left',
    xycoords='axes fraction', textcoords='offset points')

plt.tight_layout()
plt.show()

모든 별에 대해 구해보자.

In [ ]:
from photutils.aperture import CircularAperture as CAp
from photutils.aperture import CircularAnnulus as CAn

apert = CAp(pos, r=R_AP)
annul = CAn(positions=pos, r_in=4*FWHM, r_out=6*FWHM)

# since our `annul` has many elements,
mask_apert = (apert.to_mask(method='center'))
mask_annul = (annul.to_mask(method='center'))
# CAUTION!! YOU MUST USE 'center', NOT 'exact'!!!

for i in range(len(DAOfound['id'])):
    annul_weighted = mask_annul[i].multiply(hdul[0].data)
    sky_non0   = np.nonzero(annul_weighted)
    sky_pixel  = annul_weighted[sky_non0]
    msky, sky_std, nsky, nrej = sky_fit(sky_pixel, method='mode', mode_option='sex')
    #print('{0:7d}: {1:.5f} {2:.5f} {3:4d} {4:3d}'.format(idx, msky, sky_std, nsky, nrej))
    plt.errorbar([i], msky, yerr=sky_std, capsize=3, color='b')

plt.xlabel('Star ID')
plt.ylabel('msky +- sky_std')
plt.grid(ls=':')
plt.show()

## instrumental magnitude

### rdnoise and gain

CCD spec에서 readout noise 값과 gain 값을 알아보자. fits header에 들어있지 않지만, 데이터 처리하는 과정에서 fits header에 넣어 주기도 한다.

In [ ]:
rdnoise = hdul[0].header["RDNOISE"]
gain    = hdul[0].header["GAIN"]

print(rdnoise, gain)

### aperture sum

구경 안의 픽셀 값들의 합은 aperture_photometry 함수로 쉽게 구할 수 있다.

In [ ]:
from photutils.aperture import aperture_photometry as apphot
apphot_result = apphot(hdul[0].data, apert, method='center')
print(type(apphot_result))
apphot_result

### aperture area

구경의 넓이 구하기

In [ ]:
ap_area   = apert.area
ap_area

### 모든 별 측광하기

DAOfinder로 찾은 모든 별에 대하여 기기등급을 구할 수 있다.

In [ ]:
mag_ann  = np.zeros(len(apphot_result))
merr_ann = np.zeros(len(apphot_result))

#Returns magnitude from flux.
def mag_inst(flux, ferr):
    m_inst = -2.5 * np.log10(flux)
    merr   = 2.5/ np.log(10) * ferr / flux
    return m_inst, merr

fig, ax = plt.subplots()

for i in range(len(apphot_result)):
    annul_weighted = mask_annul[i].multiply(hdul[0].data)
    sky_non0   = np.nonzero(annul_weighted)
    sky_pixel  = annul_weighted[sky_non0]
    msky, sky_std, nsky, nrej = sky_fit(sky_pixel, method='mode', mode_option='sex')

    flux_star = apphot_result['aperture_sum'][i] - msky * ap_area  # total - sky

    flux_err  = np.sqrt(apphot_result['aperture_sum'][i] * gain    # Poissonian (star + sky)
                        + ap_area * rdnoise**2 # Gaussian
                        + (ap_area * (gain * sky_std))**2 / nsky )

    mag_ann[i], merr_ann[i] = mag_inst(flux_star, flux_err)
    # print(i, msky, sky_std, nsky, nrej)
    # print(i, mag_ann[idx], merr_ann[idx])
    try:
        ax.errorbar(i, mag_ann[i], yerr=merr_ann[i],
                    marker='x',
                    #ms=10,
                    capsize=3)
    except:
        continue

ax.invert_yaxis()
#ax.set_ylim(ymin=-13, ymax=-10)
plt.xlabel('Star ID')
plt.ylabel('Instrumental mag')
plt.grid(ls=':')
plt.show()


### 데이터프레임
데이터 프레임에도 저장해 보자.

In [ ]:
df["aperture_sum"] = apphot_result["aperture_sum"]

for i, row in df.iterrows():
    annul_weighted = mask_annul[i].multiply(hdul[0].data)
    sky_non0   = np.nonzero(annul_weighted)
    sky_pixel  = annul_weighted[sky_non0]
    msky, sky_std, nsky, nrej = sky_fit(sky_pixel, method='mode', mode_option='sex')

    flux_star = apphot_result['aperture_sum'][i] - msky * ap_area  # total - sky

    flux_err  = np.sqrt(apphot_result['aperture_sum'][i] * gain    # Poissonian (star + sky)
                        + ap_area * rdnoise**2 # Gaussian
                        + (ap_area * (gain * sky_std))**2 / nsky )

    mag_ann, merr_ann = mag_inst(flux_star, flux_err)
    df.loc[i, 'msky'] = msky
    df.loc[i, 'sky_std'] = sky_std
    df.loc[i, 'nsky'] = nsky
    df.loc[i, 'nrej'] = nrej
    df.loc[i, 'flux_star'] = flux_star
    df.loc[i, 'flux_err'] = flux_err
    df.loc[i, 'mag_inst'] = mag_ann
    df.loc[i, 'merr_inst'] = merr_ann

df.to_csv(f"{str(BASEPATH / save_dir_name)}/{fpath.stem}_m_inst.csv")
df

## (과제)

전처리된 파일의 기기등급을 위와 같은 방법으로 구하여 제출하시오.

In [ ]:
#(과제) 이곳에 코딩을 완성하여 제출하시오.
print(f"fpaths: {fpaths}")